In [4]:
import os

IPS = os.environ['EPICS_CA_ADDR_LIST']

if not ' 10.128.249.21' in IPS:
    os.environ['EPICS_CA_ADDR_LIST'] += ' 10.128.249.21'

print(os.environ['EPICS_CA_ADDR_LIST'])

10.0.38.59:62000 10.30.13.22 10.30.14.19 10.128.249.21


In [8]:
from epics import Motor, PV
import time
import numpy as np
import matplotlib.pyplot as plt
import glob
import re

# Energy
PGM_Energy = "IPE:A:PB04:CS1:m7"

# Detector
DVF_rixs_exitslit = "IPE:B:RIO01:9215B:ai0"
CRIO_B_Avg_Time   = "IPE:B:RIO01:PvAvgTime"

# WBS
WBS_horizontal_gap    = 'IPE:A:PB01:CS4:m7'
WBS_horizontal_offset = 'IPE:A:PB01:CS4:m8'
WBS_vertical_gap      = 'IPE:A:PB01:CS3:m7'
WBS_vertical_offset   = 'IPE:A:PB01:CS3:m8'

In [9]:
from siriuspy.devices import UE44, CurrInfoSI
from siriuspy.search import IDSearch

In [10]:
def initialize_id(beamline):
    # Search ID
    devnameid = IDSearch.conv_beamline_2_idname(beamline=beamline)
    id = UE44(devname=devnameid)

    # Disable beamline control
    id.cmd_beamline_ctrl_disable()
    print('beamline control: ', id.is_beamline_ctrl_enabled)

    # Set phase speed
    id.set_kparameter_speed(0.1)
    time.sleep(0.5)
    print('pahse speed: {:.3f} mm/s'.format(id.kparameter_speed))

    return id

def move_ue44_kparam(id:UE44, phase, timeout, verbose=False):
    id.set_kparameter(phase)
    time.sleep(0.5)
    print('KParameter-RB {:.3f} mm'.format(id.kparameter)) if verbose else 0
    if id.cmd_move_kparameter_start(timeout):
        time.sleep(0.5)
        print('Undulator is moving...') if verbose else 0
        while id.is_moving:
            time.sleep(0.1)
            print('Current phase {:.3f} mm.'.format(id.kparameter_mon), end='\r') if verbose else 0
        print('Phase {:.3f} mm reached.'.format(id.kparameter)) if verbose else 0
        return True
    else:
        print('Error while cmd_move_start.')
        return False

def move_id_kparam_robust(id:UE44, phase, timeout, maxiter=3, verbose=False):
    sucess = move_ue44_kparam(id, phase=phase, timeout=timeout, verbose=verbose)
    i=0
    while not sucess and i<maxiter:
        i += 1
        print('Trying {:0f}/{:.0f}'.format(i, maxiter)) if verbose else 0
        id.cmd_reset(timeout=timeout)
        time.sleep(0.5)
        sucess = move_ue44_kparam(id, phase=phase, timeout=timeout, verbose=verbose)
    if sucess:
        print('Movimentation done!\n')
        return True
    else:
        print('Error while moving.\n')
        return False


und = initialize_id('IPE')
currinfo = CurrInfoSI()

beamline control:  True
Could not set value of SI-11SP:ID-UE44:Velo-SP
pahse speed: 0.100 mm/s


In [22]:
und.kparameter_mon

15.906877999999992

In [ ]:
# LCH1_SP = PV('SI-11SP:PS-LCH-1:Current-SP')
# LCV1_SP = PV('SI-11SP:PS-LCV-1:Current-SP')
# LCV2_SP = PV('SI-11SP:PS-LCV-2:Current-SP')

# LCH1_Mon = PV('SI-11SP:PS-LCH-1:Current-Mon')
# LCV1_Mon = PV('SI-11SP:PS-LCV-1:Current-Mon')
# LCV2_Mon = PV('SI-11SP:PS-LCV-2:Current-Mon')

# LC_phase_mon = PV('SI-11SP:BS-IDFF-LC:IDPos-Mon')
# LC_clear_flags_cmd = PV('SI-11SP:BS-IDFF-LC:ClearFlags-Cmd')
# LC_flags_mon = PV('SI-11SP:BS-IDFF-LC:Alarms-Mon')

# LC_min_current = -3.5
# LC_max_current = +3.5

ref_kparam_LH = 15.965
ref_kparam_LH = 13.450

In [ ]:
# -------- detector --------
det = PV(DVF_rixs_exitslit)
avgtime = PV(CRIO_B_Avg_Time)

# -------- motores --------
energy = Motor(PGM_Energy)

h_gap = Motor(WBS_horizontal_gap)
h_off = Motor(WBS_horizontal_offset)
v_gap = Motor(WBS_vertical_gap)
v_off = Motor(WBS_vertical_offset)

In [ ]:
def get_flux(
    det: PV,
    crio_avg_time: PV,
    exposure: float = 0.1,
):
    crio_avg_time.put(exposure)
    time.sleep(exposure)
    flux = det.get()
    return flux


def move_slit(
    x: float,
    y: float,
    slit_off_devs: list[Motor, Motor] = [h_off, v_off],
    timeout: float = 20,
):
    slit_off_devs[0].move(x, wait=True, timeout=timeout)
    slit_off_devs[1].move(y, wait=True, timeout=timeout)

    return True


def start_search(
    posx_start: float,
    posy_start: float,
    lr: float = 4e-20,
    step_x: float = 1.5,
    step_y: float = 1.5,
    tol_pos: float = 0.05,
    gap_h: float = 0.2,
    gap_v: float = 0.2,
    exposure: float = 0.1,
    timeout: float = 20,
    iterations: int = 20,
    slit_accp_devs: list[Motor, Motor] = [h_gap, v_gap],
    slit_off_devs: list[Motor, Motor] = [h_off, v_off],
):
    """."""
    history = list()

    slit_accp_devs[0].move(gap_h, wait=True, timeout=timeout)
    slit_accp_devs[1].move(gap_v, wait=True, timeout=timeout)

    df_dx = 0
    df_dy = 0
    x0 = posx_start
    y0 = posy_start

    print("Start scanning...")

    for i in range(iterations):
        x0 += lr * df_dx
        y0 += lr * df_dy

        # Current point
        success = move_slit(
            x=x0,
            y=y0,
            slit_off_devs=slit_off_devs,
        )

        avgtime.put(exposure)
        time.sleep(exposure)
        f_0 = get_flux(
            det=det,
            crio_avg_time=avgtime,
            exposure=exposure,
        )

        posx_current = h_off.readback
        posy_current = v_off.readback
        print(f"Iteration: ({i + 1}/{iterations})")
        print(f"Calculated step: ({lr * df_dx:.5f}, {lr * df_dy:.5f}) mm")
        print(f"Pos: ({posx_current:.5f}, {posy_current:.5f}) mm")
        print(f"Detector intensity: {f_0}")
        print()

        history.append((posx_current, posy_current, f_0))
        if i > 0 and lr * df_dx < tol_pos and lr * df_dy < tol_pos:
            print(f"Finished with {i} iterations")
            break

        # Horizontal direction gradient measurement
        success = move_slit(
            x0 + step_x,
            y0,
            slit_off_devs=slit_off_devs,
        )

        avgtime.put(exposure)
        time.sleep(exposure)
        f_x = get_flux(
            det=det,
            crio_avg_time=avgtime,
            exposure=exposure,
        )

        # Vertical direction gradient measurement
        success = move_slit(
            x0,
            y0 + step_y,
            slit_off_devs=slit_off_devs,
        )

        avgtime.put(exposure)
        time.sleep(exposure)
        f_y = get_flux(
            det=det,
            crio_avg_time=avgtime,
            exposure=exposure,
        )

        df_dx = (f_x - f_0) / step_x
        df_dy = (f_y - f_0) / step_y

    print("Done!!")
    return np.array(history)

In [ ]:
# Slit gap debug

h_gap.move(0.2, wait=True)
v_gap.move(0.2, wait=True)

In [ ]:
# Slit move debug

h_off.move(0.3, wait=True)
v_off.move(0.1, wait=True)

In [ ]:
# Slit move function debug

move_slit(
    x=0.3,
    y=0.1,
    slit_off_devs=[h_off, v_off]
)

In [ ]:
# Detector intesity debug

exposure = 0.1
avgtime.put(exposure)
time.sleep(exposure)
det.get()

In [ ]:
# Get flux function debug

get_flux(
    det=det,
    crio_avg_time=avgtime,
    exposure=exposure,
)

In [ ]:
lr = 4e-20  # Needs calibrate
tol_pos = 0.05

start_search(
    posx_start=0.3,
    posy_start=0.1,
    lr=4e-20,
    step_x=0.2,
    step_y=0.2,
    tol_pos=0.05,
    gap_h=0.2,
    gap_v=0.2,
    exposure=0.1,
    slit_accp_devs=[h_gap, v_gap],
    slit_off_devs=[h_off, v_off],
    iterations=20,
    timeout=20,
)